In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# --- 辅助函数 ---
def get_clinical_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tp / (tp + fn), tn / (tn + fp)

# --- (0) 数据准备  ---
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", 
           "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]
df = pd.read_csv(url, names=columns, na_values='?')
df.dropna(inplace=True)
df["target"] = (df["target"] > 0).astype(int)

# 标准化
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(df.drop("target", axis=1)), columns=df.columns[:-1])
y = df["target"]

# 划分训练集 (70%) 和测试集 (30%)
# 注意：这里得到的是"原始完整"的 X_train 和 y_train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [4]:
# --- (1) 全监督模型训练 (Gold Standard) ---
# 使用原始划分的 y_train (包含 100% 标签，未被 Mask 为 -1)
# 模型选用与基线一致的逻辑回归，以便直接对比数据量的影响
model_full = LogisticRegression(max_iter=1000, random_state=42)

print(f"开始训练全监督模型 (样本数: {len(X_train)})...")
model_full.fit(X_train, y_train)

# --- (2) 性能评估 ---
y_pred_full = model_full.predict(X_test)
y_prob_full = model_full.predict_proba(X_test)[:, 1]

acc_full = accuracy_score(y_test, y_pred_full)
auc_full = roc_auc_score(y_test, y_prob_full)
sens_full, spec_full = get_clinical_metrics(y_test, y_pred_full)

print("\n=== 全监督模型 (100% Labeled) 最终表现 ===")
print(f"Accuracy:    {acc_full:.4f}")
print(f"AUC:         {auc_full:.4f}")
print(f"Sensitivity: {sens_full:.4f}")
print(f"Specificity: {spec_full:.4f}")

开始训练全监督模型 (样本数: 207)...

=== 全监督模型 (100% Labeled) 最终表现 ===
Accuracy:    0.8444
AUC:         0.9439
Sensitivity: 0.7857
Specificity: 0.8958
